In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window
from pyspark.sql import DataFrame
from pyspark.sql.types import DoubleType, FloatType

silver = spark.table("workspace.silver.gho_clean")

In [0]:
indicator_names = {
    "GHED_GGHE-D_pc_US_SHA2011": "gov_health_spend_per_capita_usd",
    "HIV_0000000026": "hiv_indicator",
    "MDG_0000000029": "mdg_indicator",
    "NUTRITION_ANT_HAZ_NE2": "child_stunting_prev",
    "M_Est_smk_curr_std": "smoking_prev",
    "WHOSIS_000001": "life_expectancy",
    "WHOSIS_000002": "healthy_life_expectancy",
    "BP_04": "hypertension_prev",
    "NCDMORT3070": "ncd_mortality_30_70",
    "NCD_BMI_30C": "obesity_prev",
    "NCD_GLUC_04": "high_glucose_prev",
    "NUTRITION_ANAEMIA_REPRODUCTIVEAGE_PREV": "anaemia_prev_women",
    "SDGPM25": "pm25_exposure",
    "WSH_WATER_SAFELY_MANAGED": "safe_water_pct",
    "WSH_SANITATION_SAFELY_MANAGED": "safe_sanitation_pct",
}

In [0]:
mapping = F.create_map([F.lit(x) for kv in indicator_names.items() for x in kv])
named = silver.withColumn("indicator", mapping[F.col("indicator_code")])

w = Window.partitionBy("country_code", "indicator").orderBy(F.desc("year"))
gold_latest = (
    named.withColumn("rn", F.row_number().over(w))
    .filter("rn = 1").drop("rn", "indicator_code")
)

gold_wide = (
    named.groupBy("country_code", "region", "year")
    .pivot("indicator", list(indicator_names.values()))
    .agg(F.first("numeric_value"))
)

In [0]:
def _is_missing(df: DataFrame, column: str):
    col = F.col(column)
    if isinstance(df.schema[column].dataType, (DoubleType, FloatType)):
        return col.isNull() | F.isnan(col)
    return col.isNull()


def count_missing(df: DataFrame, columns: list[str]) -> dict[str, int]:
    row = df.agg(
        *[F.sum(_is_missing(df, c).cast("int")).alias(c) for c in columns]
    ).first()
    return {c: row[c] or 0 for c in columns}


def assert_no_nulls(df: DataFrame, columns: list[str], table_name: str) -> None:
    failures = {c: n for c, n in count_missing(df, columns).items() if n > 0}
    assert not failures, f"{table_name}: nulls in required columns: {failures}"


def assert_not_all_null(df: DataFrame, columns: list[str], table_name: str) -> None:
    total = df.count()
    empty = [c for c, n in count_missing(df, columns).items() if n == total]
    assert not empty, f"{table_name}: columns entirely null: {empty}"


def report_null_rates(df: DataFrame, columns: list[str]) -> DataFrame:
    total = df.count()
    rows = [(c, n, n / total if total else 0.0) for c, n in count_missing(df, columns).items()]
    return spark.createDataFrame(rows, "column string, null_count long, null_rate double")

In [0]:
assert_no_nulls(gold_latest, ["country_code", "indicator", "year", "numeric_value"], "gho_latest")
assert_no_nulls(gold_wide, ["country_code", "year"], "gho_wide")

indicator_columns = list(indicator_names.values())
assert_not_all_null(gold_wide, indicator_columns, "gho_wide")

n_unmapped = named.filter(F.col("indicator").isNull()).select("indicator_code").distinct().count()
assert n_unmapped == 0, f"{n_unmapped} indicator_code values missing from indicator_names"

display(report_null_rates(gold_wide, indicator_columns).orderBy(F.desc("null_rate")))

In [0]:
n_dup_latest = (gold_latest.groupBy("country_code", "indicator").count()
    .filter("count > 1").count())
assert n_dup_latest == 0, f"{n_dup_latest} duplicated keys in gold_latest"

n_dup_wide = (gold_wide.groupBy("country_code", "year").count()
    .filter("count > 1").count())
assert n_dup_wide == 0, f"{n_dup_wide} duplicated keys in gold_wide"

expected_columns = 3 + len(indicator_names)
assert len(gold_wide.columns) == expected_columns, (
    f"{len(gold_wide.columns)} columns in gold_wide, expected {expected_columns}"
)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

(gold_latest.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gho_latest"))

(gold_wide.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gho_wide"))

In [0]:
latest = spark.table("workspace.gold.gho_latest")
wide = spark.table("workspace.gold.gho_wide")

assert latest.count() == gold_latest.count(), "gho_latest: row count differs from DataFrame"
assert wide.count() == gold_wide.count(), "gho_wide: row count differs from DataFrame"
assert set(wide.columns) == set(gold_wide.columns), "gho_wide: schema differs from DataFrame"

In [0]:
display(latest)

In [0]:
display(wide)